In [ ]:
from typing import TypedDict,List
from dotenv import load_dotenv
from pydantic import BaseModel,Field
from langchain_core.documents import Document
from langchain_core.prompts import ChatPromptTemplate
from langchain_google_genai import ChatGoogleGenerativeAI

from langgraph.graph import StateGraph,START,END
load_dotenv()

In [ ]:
llm=ChatGoogleGenerativeAI(model="gemini-2.5-flash")

In [ ]:
class State(TypedDict):
    question:str
    docs:List[Document]
    good_docs:List[Document]
    verdict:str
    reason:str
    strips:List[str]
    kept_strips:List[str]
    refined_context:str
    answer:str

In [ ]:
class RetrievalEvaluation(BaseModel):
    score:float=Field(description="Retrieval quality score between 0 and 1")
    reason:str=Field(description="Brief explanation for retrieval score")
    

In [ ]:
retrieval_eval_prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            """
You are a strict retrieval evaluator.

Your task is to evaluate whether the retrieved documents
are useful for answering the user's question.

Evaluate ONLY the provided question and retrieved documents.

Give a score between 0 and 1.

Scoring guidelines:

0.0 - 0.3:
Poor retrieval.
The documents are irrelevant or provide almost no useful
information for answering the question.

0.3 - 0.7:
Uncertain or partially useful retrieval.
Some information is relevant, but the documents may not
provide enough information to answer the question completely.

0.7 - 1.0:
Good retrieval.
The documents are directly relevant and contain enough
information to answer the question.

Consider:

- Relevance to the question
- Coverage of the question
- Specificity of the information
- Whether the retrieved documents contain useful evidence

Do not use outside knowledge.

Return ONLY the structured evaluation.
"""
        ),
        (
            "human",
            """
Question:
{question}

Retrieved Documents:
{context}
"""
        )
    ]
)

In [ ]:
retrieval_eval_chain=(retrieval_eval_prompt |llm.with_structured_output(RetrievalEvaluation))

In [ ]:
def retrieve(state:State)->dict:
    question=state["question"]
    docs=retriever.invoke(question)
    return {
        "docs":docs
    }


In [ ]:
def evaluate_reterieval(state:State)->dict:
    question=state["question"]
    context="\n\n".join(
        f"Document {i+1}:\n {doc.page_content}"
        for i,doc in enumerate(state["docs"])
    )
    result=retrieval_eval_chain.invoke({
        "question":question,
        "context":context
    })
    score=max(0.0,min(1.0,result.score))
    if score>0.7:
        verdict="good"

    elif score<=0.3:
        verdict="poor"

    else:
        vedict="uncertain"
    return {
        "verdict":verdict,
        "reason":reason
        }

In [ ]:
def route_retreival(state:State):
    if state["verdict"]=="good":
        return "good"
    elif state["verdict"]=="poor":
        return "poor"
    else:
        return "uncertain"
    

In [ ]:
def keep_documents(state:State)->dict:
    return{
        "good_docs":state["docs"]
    }

In [ ]:
def handle_uncertain(state: State) -> dict:

    # For this iteration, we keep the retrieved documents.
    # In a later CRAG iteration, you can add reranking,
    # query rewriting, or another retrieval strategy here.

    return {
        "good_docs": state["docs"]
    }

In [ ]:
def handle_poor(state:State)->dict:
    return {
        "good_docs": state["docs"]
    }

In [ ]:
import re


def decompose_to_sentences(text: str) -> list[str]:

    text = re.sub(
        r"\s+",
        " ",
        text
    ).strip()

    sentences = re.split(
        r"(?<=[.!?])\s+",
        text
    )

    return [
        sentence.strip()
        for sentence in sentences
        if len(sentence.strip()) > 20
    ]


In [ ]:
class KeepOrDrop(BaseModel):

    keep: bool = Field(
        description=(
            "True if the sentence directly helps answer "
            "the question, otherwise False."
        )
    )


filter_prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            """
You are a strict relevance filter.

Determine whether the given sentence directly helps answer
the user's question.

Return keep=true ONLY if the sentence contains information
that is directly useful for answering the question.

Return keep=false if the sentence is:

- Irrelevant
- Only loosely related
- Too vague
- Not useful for answering the question

Do not use outside knowledge.

Evaluate ONLY the provided sentence.

Return structured output only.
"""
        ),
        (
            "human",
            """
Question:
{question}

Sentence:
{sentence}
"""
        )
    ]
)

In [ ]:
filter_chain = (
    filter_prompt
    | llm.with_structured_output(KeepOrDrop)
)

In [ ]:
def refine(state: State) -> dict:

    question = state["question"]

    context = "\n\n".join(
        doc.page_content
        for doc in state["good_docs"]
    ).strip()

    strips = decompose_to_sentences(
        context
    )

    kept_strips = []

    for sentence in strips:

        result = filter_chain.invoke(
            {
                "question": question,
                "sentence": sentence
            }
        )

        if result.keep:
            kept_strips.append(sentence)

    refined_context = "\n\n".join(
        kept_strips
    ).strip()

    return {
        "strips": strips,
        "kept_strips": kept_strips,
        "refined_context": refined_context
    }

In [ ]:
def generate(state: State) -> dict:

    question = state["question"]

    context = state["refined_context"]

    prompt = f"""
Answer the question using ONLY the provided context.

Question:
{question}

Context:
{context}

Rules:

- Do not invent information.
- Do not use outside knowledge.
- If the context does not contain enough information,
  clearly say that the available context is insufficient.
"""

    response = llm.invoke(prompt)

    return {
        "answer": response.content
    }

In [ ]:
g = StateGraph(State)


# Nodes
g.add_node(
    "retrieve",
    retrieve
)

g.add_node(
    "evaluate",
    evaluate_retrieval
)

g.add_node(
    "keep_documents",
    keep_documents
)

g.add_node(
    "handle_uncertain",
    handle_uncertain
)

g.add_node(
    "handle_poor",
    handle_poor
)

g.add_node(
    "refine",
    refine
)

g.add_node(
    "generate",
    generate
)


In [ ]:
g.add_edge(
    START,
    "retrieve"
)

g.add_edge(
    "retrieve",
    "evaluate"
)


g.add_conditional_edges(
    "evaluate",
    route_retrieval,
    {
        "good": "keep_documents",
        "uncertain": "handle_uncertain",
        "poor": "handle_poor"
    }
)


g.add_edge(
    "keep_documents",
    "refine"
)

g.add_edge(
    "handle_uncertain",
    "refine"
)

g.add_edge(
    "handle_poor",
    "refine"
)

g.add_edge(
    "refine",
    "generate"
)

g.add_edge(
    "generate",
    END
)


In [ ]:
app = g.compile()

In [ ]:
result = app.invoke(
    {
        "question": "What is the main concept discussed in the book?",
        "docs": [],
        "good_docs": [],
        "verdict": "",
        "reason": "",
        "strips": [],
        "kept_strips": [],
        "refined_context": "",
        "answer": ""
    }
)


print("VERDICT:", result["verdict"])
print("REASON:", result["reason"])
print("\nANSWER:\n")
print(result["answer"])